# Task 1: 数据质量检查与基础分析

**负责人**: _________  
**日期**: 2026-01-22  
**预计时长**: 3-4 小时

## 任务目标
1. 检查数据完整性和质量
2. 识别不同市场状态（牛市/熊市/震荡）
3. 计算基础统计指标
4. 为后续分析提供数据基础

## 1. 环境准备

In [ ]:
import sys
sys.path.append('..')  # 添加父目录到路径

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from utils import load_data, calculate_max_drawdown, save_results_to_csv

# 设置绘图样式
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# 显示所有列
pd.set_option('display.max_columns', None)

print("✅ 环境准备完成")

## 2. 数据加载与初步检查

In [ ]:
# 加载 AAPL 数据
df = load_data('aapl', data_dir='../data')

print(f"数据时间范围: {df['date'].min()} 到 {df['date'].max()}")
print(f"总记录数: {len(df):,}")
print(f"\n数据列: {list(df.columns)}")
print(f"\n前 5 行:")
df.head()

In [ ]:
# 基本信息
print("数据类型和内存使用:")
df.info()

## 3. 数据质量检查

In [ ]:
# 3.1 缺失值检查
print("缺失值统计:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    '缺失数量': missing,
    '缺失比例(%)': missing_pct
})
print(missing_summary[missing_summary['缺失数量'] > 0])

if missing.sum() == 0:
    print("\n✅ 无缺失值")
else:
    print(f"\n⚠️ 发现 {missing.sum()} 个缺失值")

In [ ]:
# 3.2 异常值检查
print("\n价格数据统计摘要:")
print(df[['open', 'high', 'low', 'close', 'volume']].describe())

# 检查异常
anomalies = []

# 负价格
if (df[['open', 'high', 'low', 'close']] < 0).any().any():
    anomalies.append("发现负价格")

# 零成交量
if 'volume' in df.columns:
    zero_vol = (df['volume'] == 0).sum()
    if zero_vol > 0:
        anomalies.append(f"发现 {zero_vol} 天零成交量")

# 价格逻辑异常 (high < low)
if (df['high'] < df['low']).any():
    anomalies.append("发现最高价 < 最低价的异常")

if anomalies:
    print("\n⚠️ 发现异常:")
    for a in anomalies:
        print(f"  - {a}")
else:
    print("\n✅ 未发现明显异常")

In [ ]:
# 3.3 重复日期检查
duplicate_dates = df['date'].duplicated().sum()
if duplicate_dates > 0:
    print(f"⚠️ 发现 {duplicate_dates} 个重复日期")
    print(df[df['date'].duplicated(keep=False)].sort_values('date'))
else:
    print("✅ 无重复日期")

## 4. 计算基础收益指标

In [ ]:
# 计算收益率
df['daily_return'] = df['close'].pct_change()
df['log_return'] = np.log(df['close'] / df['close'].shift(1))
df['cumulative_return'] = (1 + df['daily_return']).cumprod()

# 整体统计
total_days = len(df)
trading_years = total_days / 252
total_return = df['cumulative_return'].iloc[-1] - 1
annual_return = (1 + total_return) ** (1 / trading_years) - 1
annual_vol = df['daily_return'].std() * np.sqrt(252)
max_dd = calculate_max_drawdown(df['cumulative_return'])

print("\n=" * 60)
print("整体表现统计 (Buy & Hold)")
print("=" * 60)
print(f"交易天数: {total_days:,}")
print(f"交易年数: {trading_years:.2f}")
print(f"总收益率: {total_return:.2%}")
print(f"年化收益率: {annual_return:.2%}")
print(f"年化波动率: {annual_vol:.2%}")
print(f"夏普比率: {annual_return / annual_vol:.3f}")
print(f"最大回撤: {max_dd:.2%}")
print(f"Calmar 比率: {annual_return / abs(max_dd):.3f}")
print("=" * 60)

## 5. 按年份分析

In [ ]:
# 添加年份列
df['year'] = df['date'].dt.year

# 按年份统计
yearly_stats = df.groupby('year').agg({
    'close': ['first', 'last', 'min', 'max'],
    'daily_return': ['mean', 'std', 'count'],
    'volume': 'mean'
})

# 计算年度收益
yearly_stats.columns = ['_'.join(col).strip() for col in yearly_stats.columns.values]
yearly_stats['annual_return'] = (yearly_stats['close_last'] / yearly_stats['close_first'] - 1) * 100
yearly_stats['annual_vol'] = yearly_stats['daily_return_std'] * np.sqrt(252) * 100
yearly_stats['sharpe'] = yearly_stats['annual_return'] / yearly_stats['annual_vol']

# 选择关键列
display_cols = ['annual_return', 'annual_vol', 'sharpe', 'daily_return_count']
yearly_display = yearly_stats[display_cols].round(2)
yearly_display.columns = ['年度收益(%)', '年化波动率(%)', '夏普比率', '交易天数']

print("\n年度表现统计:")
print(yearly_display)

# 保存结果
save_results_to_csv(yearly_display, 'task1_yearly_stats.csv')

## 6. 市场状态识别

In [ ]:
# 定义市场状态
# 牛市: 年度收益 > 15%, 夏普 > 0.5
# 熊市: 年度收益 < -5%
# 震荡: 其他

def classify_market_state(row):
    if row['annual_return'] > 15 and row['sharpe'] > 0.5:
        return '牛市'
    elif row['annual_return'] < -5:
        return '熊市'
    else:
        return '震荡'

yearly_stats['market_state'] = yearly_stats.apply(classify_market_state, axis=1)

# 统计各市场状态
print("\n市场状态分布:")
print(yearly_stats['market_state'].value_counts())

# 显示分类结果
market_summary = yearly_stats[['annual_return', 'annual_vol', 'sharpe', 'market_state']].round(2)
market_summary.columns = ['年度收益(%)', '年化波动率(%)', '夏普比率', '市场状态']
print("\n各年份市场状态:")
print(market_summary)

# 保存市场状态
save_results_to_csv(market_summary, 'task1_market_states.csv')

## 7. 可视化分析

In [ ]:
# 7.1 价格走势图
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# 收盘价
axes[0].plot(df['date'], df['close'], linewidth=1.5, color='steelblue')
axes[0].set_title('AAPL 历史价格走势', fontsize=14, fontweight='bold')
axes[0].set_ylabel('收盘价 ($)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# 累计收益
axes[1].plot(df['date'], (df['cumulative_return'] - 1) * 100, 
             linewidth=1.5, color='green')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('累计收益率 (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('日期', fontsize=12)
axes[1].set_ylabel('收益率 (%)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../diagnosis/figures/task1_price_trends.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 图表已保存到 diagnosis/figures/task1_price_trends.png")

In [ ]:
# 7.2 年度收益柱状图
fig, ax = plt.subplots(figsize=(14, 6))

colors = yearly_stats['annual_return'].apply(
    lambda x: 'green' if x > 0 else 'red'
)

bars = ax.bar(yearly_stats.index, yearly_stats['annual_return'], 
              color=colors, alpha=0.7, edgecolor='black')

# 添加数值标签
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%',
            ha='center', va='bottom' if height > 0 else 'top',
            fontsize=9)

ax.axhline(y=0, color='black', linewidth=1)
ax.set_title('AAPL 年度收益率', fontsize=14, fontweight='bold')
ax.set_xlabel('年份', fontsize=12)
ax.set_ylabel('年度收益率 (%)', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../diagnosis/figures/task1_annual_returns.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 图表已保存到 diagnosis/figures/task1_annual_returns.png")

In [ ]:
# 7.3 收益分布直方图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 日收益分布
axes[0].hist(df['daily_return'].dropna() * 100, bins=100, 
             color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].set_title('日收益率分布', fontsize=14, fontweight='bold')
axes[0].set_xlabel('日收益率 (%)', fontsize=12)
axes[0].set_ylabel('频数', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')

# Q-Q 图
from scipy import stats
stats.probplot(df['daily_return'].dropna(), dist="norm", plot=axes[1])
axes[1].set_title('Q-Q 图（正态性检验）', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../diagnosis/figures/task1_return_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 图表已保存到 diagnosis/figures/task1_return_distribution.png")

## 8. 总结与发现

In [ ]:
print("\n" + "="*80)
print("Task 1 执行总结")
print("="*80)

print("\n✅ 完成的工作:")
print("  1. 数据质量检查 - 无缺失值和重大异常")
print(f"  2. 分析了 {len(df):,} 天的交易数据 ({trading_years:.1f} 年)")
print(f"  3. 识别了 {len(yearly_stats)} 个年份的市场状态")
print("  4. 生成了 3 张可视化图表")

print("\n📊 关键发现:")
print(f"  - 总体 Buy & Hold 收益: {total_return:.2%}")
print(f"  - 年化收益率: {annual_return:.2%}")
print(f"  - 最大回撤: {max_dd:.2%}")
print(f"  - 牛市年份: {(yearly_stats['market_state'] == '牛市').sum()} 个")
print(f"  - 熊市年份: {(yearly_stats['market_state'] == '熊市').sum()} 个")
print(f"  - 震荡年份: {(yearly_stats['market_state'] == '震荡').sum()} 个")

print("\n💡 对后续分析的建议:")
print("  1. 可以按市场状态分别测试策略表现")
print("  2. 重点关注熊市/震荡市的策略有效性")
print("  3. 考虑动态调整策略参数以适应不同市场")

print("\n📁 交付物:")
print("  - task1_yearly_stats.csv")
print("  - task1_market_states.csv")
print("  - task1_price_trends.png")
print("  - task1_annual_returns.png")
print("  - task1_return_distribution.png")

print("\n" + "="*80)
print("✅ Task 1 完成！")
print("="*80)

---

## 下一步

- [ ] 将本 notebook 提交到 Git
- [ ] 在团队会议上分享关键发现
- [ ] 为 Task 2（因子分析）提供清洗后的数据
- [ ] 更新 `DIAGNOSIS_TASKS.md` 中的完成状态